In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import now
from delta.exceptions import ConcurrentAppendException
import time
from datetime import datetime
import pyspark.sql.functions as f
from concurrent.futures import ThreadPoolExecutor

In [0]:
class extraccion:

    def consulta_csv(nombre_archivo):
        """
        Funcion para leer un archivo csv.
        Args:
            nombre_archivo (str): nombre del archivo csv.
        Returns:
            df (DataFrame): DataFrame con con los datos del archivo csv.
        """
        try:
            df = spark.read.format("csv").options(
            header=True,
            inferSchema=True
            ).load(f"/Workspace/Users/martin902503@gmail.com/retail_nova/data/{nombre_archivo}.csv")
        except Exception as e:
            print(f"Error funcion: consulta_csv, no se pudo leer el archivo {nombre_archivo}.csv \n Error: {e}")
        return df
    
    def consulta_delta(tabla, capa):
        """
        Funcion para leer una tabla delta.
        Args:
            tabla (str): nombre de la tabla delta.
            capa (str): capa donde se encuentra la tabla delta.
        Returns:
            df (DataFrame): DataFrame con con los datos de la tabla delta.
        """
        try:
            df = spark.sql(f"SELECT * FROM Workspace.{capa}.{tabla}")
        except Exception as e:
            print(f"Error funcion: consulta_delta, no se pudo leer la tabla delta: {tabla} \n Error: {e}")
        return df
    
    

In [0]:
class carga:

    def carga_delta(id_proceso, df, tabla, capa, flag_dependencia, partitionBy=None):
        """
        Funcion para cargar un DataFrame en delta.
        Args:
            df (DataFrame): DataFrame a cargar.
            tabla (str): nombre de la tabla delta a cargar.
            capa (str): capa donde se va a cargar la tabla delta.
            partitionBy (str): columna por la que se quiere particionar el DataFrame.
        """
        try:
            if capa == 'bronze':
                writer = df.write.mode("overwrite").format("delta")
                if partitionBy:
                    writer = writer.partitionBy(partitionBy)
                writer.option("overwriteSchema", "true").saveAsTable(f"Workspace.{capa}.{tabla}")
                configuracion.actualizacion_proceso(capa, id_proceso)
                print(f"Info: La tabla {tabla}, se ha cargado correctamente en capa {capa}.")

            elif capa == 'silver':
                if flag_dependencia == True:
                    df = configuracion.configuracion_proceso(capa, id_proceso)
                    for i in df.collect():
                        path = i['path']

                    dbutils.notebook.run(path, 600, {"fecha_procesado": str(configuracion.getdate()), "id_proceso": str(id_proceso)})
                    configuracion.actualizacion_proceso(capa, id_proceso)
                    print(f"Info: La tabla {tabla}, se ha cargado correctamente en capa {capa}")
                else:
                    writer = df.write.mode("overwrite").format("delta")
                    if partitionBy:
                        writer = writer.partitionBy(partitionBy)
                    writer.option("overwriteSchema", "true").saveAsTable(f"Workspace.{capa}.{tabla}")
                    configuracion.actualizacion_proceso(capa, id_proceso)
                    print(f"Info: La tabla {tabla}, se ha cargado correctamente en capa {capa}.")    

            elif capa == 'gold':
                df = configuracion.configuracion_proceso(capa, id_proceso)

                for i in df.collect():
                    for x in i['tabla_dependencia']:
                        input_table = f"{x}"

                    output_table = f"workspace.{capa}.{i['nombre_archivo']}"
                    llave_negocio = i['atributos_llave']
                
                carga.carga_tabla_gold_incremental(input_table, output_table, llave_negocio)
                configuracion.actualizacion_proceso(capa, id_proceso)
                print(f"Info: La tabla {tabla}, se ha cargado correctamente en capa {capa}.")

        except Exception as e:
            print(f"Error funcion: carga_delta, no se pudo cargar la tabla {tabla} en la capa {capa}: \n Error: {e}")

    def create_tabla_gold(input_table, output_table):
        try:
            input_df = spark.table(input_table)

            string_schema = ','.join([f'{x.jsonValue()["name"]} {x.jsonValue()["type"]}' for x in input_df.schema.fields])
            string_schema = string_schema.replace(',fecha_procesado date', '').replace(',fecha_procesado timestamp', '')

        except Exception as e:
            print("Se presento un error en la creacion de la tabla: ", output_table, e)

        return f"""
                create table if not exists {output_table}(
                    {string_schema},
                    fecha_cargado TIMESTAMP)
                """

    def return_keys(llave_negocio):
        try:
            campo_max = len(llave_negocio)-1
            bussiness_keys = ""
            for x in enumerate(llave_negocio):
                iterador = int(x[0])
                campo = str(x[1])

                data_concat = 'A.' + campo + ' = ' + 'B.' + campo 

                if iterador < campo_max:
                    bussiness_keys += data_concat 
                    bussiness_keys += " AND " 

                elif iterador == campo_max:
                    bussiness_keys += data_concat 

        except Exception as e:
            print("Ocurrio un error en la generacion de las llaves de negocio:", e)

        return bussiness_keys

    def carga_tabla_gold_incremental(input_table, output_table, llave_negocio):
        try:
            llave_negocio = llave_negocio.split(",")
            if not spark.catalog.tableExists(f"{input_table}"):
                print(f"La tabla de capa silver: {input_table}, no existe")

            elif not spark.catalog.tableExists(f"{output_table}"): 
                spark.sql(carga.create_tabla_gold(input_table, output_table))   

            df_input = spark.table(input_table)
            df_output = spark.table(output_table)   
            # Valido llaves de negocio mapeadas en silver
            input_keys = []
            for i in df_input.columns:
                for x in llave_negocio:
                    if i == x:
                        input_keys.append(i)    

            # Valido llaves de negocio mapeadas en gold
            output_keys = []
            for i in df_output.columns:
                for x in llave_negocio:
                    if i == x:
                        output_keys.append(i)   

            # Valido que las llaves de negocio sean las mismas
            if input_keys.sort() == llave_negocio.sort() and output_keys.sort() == llave_negocio.sort():
                flag_key = True
                print(f"El flag de llaves de negocio es: {flag_key}")
            else:
                flag_key = False
                print(f"El flag de llaves de negocio es: {flag_key}")   

            if flag_key == True:
                # se obtiene la data de las diferentes llaves de negocio en silver y gold
                data_origen = df_input.select(llave_negocio).dropDuplicates()
                data_destino = df_output.select(llave_negocio).dropDuplicates()  

                # se obtiene la data que no existe en gold
                except_data = data_origen.exceptAll(data_destino)  

                # retorna las llaves de negocio para el join
                keys_join = carga.return_keys(llave_negocio)  

                df_input.createOrReplaceTempView("vw_data_input")
                except_data.createOrReplaceTempView("vw_except_data")   

                # se aplica el innier join y se obtiene la data a insertar en gold
                data_merge = spark.sql(f"""
                                    SELECT A.*
                                    FROM vw_data_input AS A
                                    INNER JOIN vw_except_data AS B
                                    ON {keys_join}
                                    """)

                fecha_procesado = str(configuracion.getdate())
                data_merge = data_merge.withColumn("fecha_procesado", f.lit(fecha_procesado).cast("timestamp"))
                cont = data_merge.count()
                data_merge.write.option("mergeSchema", "true").format("delta").mode("append").saveAsTable(output_table)
                print(f"Se insertaron {cont} registros en la tabla gold: {output_table}")
            else:
                print(f"No se puede continuar con la ejecucion, las llaves de negocio: {llave_negocio} no coinciden en silver y gold")
        except Exception as e:
            print(f"Error funcion: carga_tabla_gold_incremental, error en la carga de la tabla: {e}")

    def orquesta_procesos(capa, id):
        try:
            metadata = configuracion.configuracion_proceso(capa, id)
    
            if metadata.count() != 0:
                if capa == 'bronze':
                    def process_table(i):
                        tabla = i["nombre_archivo"]
                        partitionBy = i["campo_particion"]
                        id_proceso = i["id"]
                        if i["flag_activo"] == True and i['flag_procesado'] == False:
                            data = extraccion.consulta_csv(tabla)
                            print(f"Info: Cargando en la capa {capa}, la tabla {tabla} .... \n")
                            carga.carga_delta(id_proceso, data, tabla, capa, flag_dependencia=False, partitionBy=None)
    
                    rows = metadata.collect()
                    with ThreadPoolExecutor() as executor:
                        executor.map(process_table, rows)
    
                elif capa == 'silver':
                    def process_dependency(i):
                        id = i['id']
                        df = metadata.filter(f"id = '{id}'").select(
                            "*",
                            explode("tabla_dependencia").alias("tabla_dependencia_exploded")
                        )
    
                        if df.count() > 1:
                            print("\n")
                            print(f"Info: La tabla {capa}.{i['nombre_archivo']} tiene mas de una dependencia:")
    
                            resultado_estatus = []
                            for j in df.collect():
                                tabla_dependencia = j["tabla_dependencia_exploded"]
                                tabla_spt = tabla_dependencia.split(".")
                                capa_dependencia = tabla_spt[1]
                                tabla = tabla_spt[2] 
    
                                flag = configuracion.valida_tabla_procesada(capa_dependencia, tabla)
                                if flag == True:
                                    resultado_estatus.append([capa_dependencia,tabla,flag])
    
                                print(f"Info: Tabla {capa_dependencia}.{tabla}, flag_procesado: {flag}")
    
                            if df.count() == len(resultado_estatus):
                                flag_dependencia = True
                                print("\n")
                                print(f"Info: Todas las dependencias de la tabla {capa}.{i['nombre_archivo']} estan procesadas")
                                data = extraccion.consulta_delta(tabla, capa_dependencia)
                                print(f"Info: Cargando en la capa {capa}, la tabla {i['nombre_archivo']} .... \n")
                                carga.carga_delta(id, data, tabla, capa, flag_dependencia, partitionBy=None)
    
                        else:
                            print("\n")
                            print(f"Info: La tabla {capa}.{i['nombre_archivo']} tiene solo una dependencia:")
    
                            resultado_estatus = []
                            for j in df.collect():
                                tabla_dependencia = j["tabla_dependencia_exploded"]
                                tabla_spt = tabla_dependencia.split(".")
                                capa_dependencia = tabla_spt[1]
                                tabla = tabla_spt[2] 
    
                                flag = configuracion.valida_tabla_procesada(capa_dependencia, tabla)
                                if flag == True:
                                    resultado_estatus.append([capa_dependencia,tabla,flag])
    
                                print(f"Info: Tabla {capa_dependencia}.{tabla}, flag_procesado: {flag}")
    
                            if df.count() == len(resultado_estatus):
                                flag_dependencia = False
                                print("\n")
                                print(f"Info: Todas las dependencias de la tabla {capa}.{i['nombre_archivo']} estan procesadas")
                                data = extraccion.consulta_delta(tabla, capa_dependencia)
                                print(f"Info: Cargando en la capa {capa}, la tabla {tabla} .... \n")
                                carga.carga_delta(id, data, tabla, capa, flag_dependencia, partitionBy=None)
    
                    rows = metadata.collect()
                    with ThreadPoolExecutor() as executor:
                        executor.map(process_dependency, rows)
    
                elif capa == 'gold':
                    def process_table(i):
                        tabla = i["nombre_archivo"]
                        id_proceso = i["id"]
                        print(id_proceso)
    
                        if i["flag_activo"] == True and i['flag_procesado'] == False:
                            # Genera un DataFrame vacío con las mismas columnas que metadata
                            empty_schema = metadata.schema
                            data = spark.createDataFrame([], empty_schema)
                            print(f"Info: Cargando en la capa {capa}, la tabla {tabla} .... \n")
                            carga.carga_delta(id_proceso, data, tabla, capa, flag_dependencia=False, partitionBy=None)
    
                    rows = metadata.collect()
                    with ThreadPoolExecutor() as executor:
                        executor.map(process_table, rows)
    
                else:
                    print("Error: No esta llegando el valor de la capa correctamente, debe ser 'bronze', 'silver' o 'gold'.")
            else:
                print("Warning: No existen tablas habilitadas en la metadata.")
        except Exception as e:
            print(f"Error funcion: orquesta_procesos, no se pudo completar la orquestacion de procesos, error: {e}")

    

In [0]:
class configuracion:

    def configuracion_proceso(capa, id):
        try:
            if id == "":
                df = spark.sql("SELECT * FROM workspace.configuration.configuracion_metadata WHERE capa = '{}' AND flag_activo = true AND flag_procesado = false".format(capa))
            else:
                df = spark.sql("SELECT * FROM workspace.configuration.configuracion_metadata WHERE id IN ({}) AND flag_activo = true AND flag_procesado = false".format(id))
        except Exception as e:
            print(f"Error funcion: configuracion_proceso, no se pudo leer la tabla configuracion_metadata \n Error: {e}")
        return df

    def update_with_retry(spark, sql_stmt, retries=3):
        for i in range(retries):
            try:
                spark.sql(sql_stmt)
                return
            except ConcurrentAppendException:
                if i == retries - 1:
                    raise
                time.sleep(2)
    
    def actualizacion_proceso(capa, id_proceso):
        try:
            sql_stmt = f"UPDATE workspace.configuration.configuracion_metadata SET flag_procesado = true, fecha_procesado = current_timestamp() WHERE id = {id_proceso}"
            configuracion.update_with_retry(spark, sql_stmt, retries=3)
        except Exception as e:
            print(f"Error funcion: actualizacion_proceso, no se pudo actualizar la tabla configuracion_metadata: {e}")

    def valida_tabla_procesada(capa, tabla):
        try:
            df = spark.sql("SELECT nombre_archivo, flag_procesado FROM workspace.configuration.configuracion_metadata WHERE capa = '{}' AND nombre_archivo = '{}'".format(capa, tabla))
            
            for i in df.collect():
                if i.flag_procesado == True:
                    return True
                else:
                    return False
        
        except Exception as e:
            print(f"Error funcion: valida_tabla_procesada, no se pudo leer la tabla configuracion_metadata: {e}")
            return None
        
    def getdate():
        """
        Retorna la fecha y hora actual en formato datetime.
        """
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")